# 11 · WorldPop population 2026 — India 100 m, total and school-age bands

**Source:** WorldPop, *Global2* release R2025A (University of Southampton), constrained 100 m (3 arc-second)
grids for India, year 2026. Bondarenko et al. (2025), DOI 10.5258/SOTON/WP00841. **Licence:** CC BY 4.0.

**Files** (each about 0.78 GB, saved in `data/raw/worldpop/`):
- `ind_pop_2026_CN_100m_R2025A_v1.tif` — total population per cell
- `ind_t_05 / t_10 / t_15_2026_CN_100m_R2025A_v1.tif` — both sexes aged 5–9, 10–14, 15–19 (school-age demand;
  the levels' exact ages are taken from these bands assuming an even spread of ages inside a band)

Each file is also cut to Kerala (`data/raw/worldpop/kerala/`) for the Kerala case study.
Note (project notes): WorldPop India is modelled from 35 state totals (Census 2011 + projections), so 2026
values are estimates, not counts.

Downloads are checked against the server's file size; a file already saved is not downloaded again, and the
notebook stops if less than 5 GB of disk would remain.

## Imports and settings

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))
import os
if "postgis" in os.environ.get("PROJ_LIB", "").lower():  # an old PROJ database from a PostGIS install breaks rasterio
    os.environ.pop("PROJ_LIB")
import json
import shutil
import time

import numpy as np
import rasterio
import requests
from rasterio.mask import mask

OUT_DIR = PROJECT / "data" / "raw" / "worldpop"
BASE = "https://data.worldpop.org/GIS/"
# WorldPop Global2 R2025A, India, 2026, 100 m (3 arc-seconds), constrained; t = both sexes, 05 = ages 5-9 etc.
FILES = {
    "ind_pop_2026_CN_100m_R2025A_v1.tif": "Population/Global_2015_2030/R2025A/2026/IND/v1/100m/constrained/",
    "ind_t_05_2026_CN_100m_R2025A_v1.tif": "AgeSex_structures/Global_2015_2030/R2025A/2026/IND/v1/100m/constrained/",
    "ind_t_10_2026_CN_100m_R2025A_v1.tif": "AgeSex_structures/Global_2015_2030/R2025A/2026/IND/v1/100m/constrained/",
    "ind_t_15_2026_CN_100m_R2025A_v1.tif": "AgeSex_structures/Global_2015_2030/R2025A/2026/IND/v1/100m/constrained/",
}
ADM1 = PROJECT / "data" / "raw" / "boundaries" / "geoBoundaries-IND-ADM1.geojson"
HEADERS = {"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"}

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

### `download()`

In [3]:
def download(url, f, min_free_gb=5):
    """Download once. A finished file keeps its name; an interrupted one stays *.part and is fetched again."""
    f.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(4):  # servers can be slow to answer; wait and retry
        try:
            size = int(requests.head(url, headers=HEADERS, allow_redirects=True, timeout=300).headers.get("content-length", 0))
            break
        except requests.RequestException as e:
            print("  server slow, retrying:", type(e).__name__)
            time.sleep(60)
    else:
        raise SystemExit(f"no answer from {url}")
    if f.exists() and (size == 0 or f.stat().st_size == size):
        print(f"already saved: {f.name} ({f.stat().st_size / 1e6:.0f} MB)")
        return f
    free = shutil.disk_usage(f.parent).free
    if free - size < min_free_gb * 1e9:
        raise SystemExit(f"not enough disk for {f.name}: {size / 1e9:.2f} GB needed, {free / 1e9:.1f} GB free")
    part = f.with_name(f.name + ".part")
    with requests.get(url, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(part, "wb") as out:
            shutil.copyfileobj(r.raw, out, length=1 << 20)
    if size and part.stat().st_size != size:
        raise SystemExit(f"{f.name}: got {part.stat().st_size} bytes, expected {size}")
    part.replace(f)
    print(f"downloaded: {f.name} ({size / 1e6:.0f} MB)")
    return f

### `clip_kerala()`

In [4]:
def clip_kerala(src):
    """Cut a raster to the Kerala state polygon (geoBoundaries ADM1); only the window around Kerala is read."""
    out = OUT_DIR / "kerala" / src.name.replace("ind_", "kerala_")
    if out.exists():
        return out
    kerala = [f["geometry"] for f in json.loads(ADM1.read_text(encoding="utf-8"))["features"]
              if f["properties"]["shapeName"] == "Kerala"]
    with rasterio.open(src) as r:
        data, transform = mask(r, kerala, crop=True)
        meta = r.meta | {"height": data.shape[1], "width": data.shape[2], "transform": transform,
                         "compress": "deflate", "tiled": True}
    out.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(out, "w", **meta) as w:
        w.write(data)
    return out

### `raster_sum()`

In [5]:
def raster_sum(f):
    """Sum of all valid cells, read block by block (an India raster does not fit in memory)."""
    total = 0.0
    with rasterio.open(f) as r:
        for _, win in r.block_windows(1):
            a = r.read(1, window=win, masked=True)
            total += float(a.sum()) if a.count() else 0.0
    return total

### `run_all()`

In [6]:
def run_all():
    for name, folder in FILES.items():
        f = download(BASE + folder + name, OUT_DIR / name)
        k = clip_kerala(f)
        print(f"  Kerala clip: {k.name}")

## Run

About 3.1 GB of downloads.

In [7]:
run_all()

downloaded: ind_pop_2026_CN_100m_R2025A_v1.tif (784 MB)


  Kerala clip: kerala_pop_2026_CN_100m_R2025A_v1.tif


downloaded: ind_t_05_2026_CN_100m_R2025A_v1.tif (782 MB)


  Kerala clip: kerala_t_05_2026_CN_100m_R2025A_v1.tif


downloaded: ind_t_10_2026_CN_100m_R2025A_v1.tif (781 MB)


  Kerala clip: kerala_t_10_2026_CN_100m_R2025A_v1.tif


downloaded: ind_t_15_2026_CN_100m_R2025A_v1.tif (782 MB)


  Kerala clip: kerala_t_15_2026_CN_100m_R2025A_v1.tif


## Check

Grid size, resolution and population totals (India and Kerala).

In [8]:
for name in FILES:
    with rasterio.open(OUT_DIR / name) as r:
        print(f"{name}: {r.width} x {r.height} cells, {r.res[0] * 3600:.1f} arc-seconds, {r.dtypes[0]}, CRS {r.crs.to_epsg()}")
    k = OUT_DIR / "kerala" / name.replace("ind_", "kerala_")
    print(f"   India total {raster_sum(OUT_DIR / name) / 1e6:9.2f} million | Kerala {raster_sum(k) / 1e6:6.2f} million")

ind_pop_2026_CN_100m_R2025A_v1.tif: 35040 x 34507 cells, 3.0 arc-seconds, float32, CRS 4326


   India total   1470.30 million | Kerala  37.42 million
ind_t_05_2026_CN_100m_R2025A_v1.tif: 35040 x 34507 cells, 3.0 arc-seconds, float32, CRS 4326


   India total    117.15 million | Kerala   2.03 million
ind_t_10_2026_CN_100m_R2025A_v1.tif: 35040 x 34507 cells, 3.0 arc-seconds, float32, CRS 4326


   India total    122.58 million | Kerala   2.25 million
ind_t_15_2026_CN_100m_R2025A_v1.tif: 35040 x 34507 cells, 3.0 arc-seconds, float32, CRS 4326


   India total    126.42 million | Kerala   2.38 million
